In [1]:
from embedder import Embedder

embed = Embedder()

2026-06-30 18:25:38.448289603 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [2]:
# Question 1
query = 'How does approximate nearest neighbor search work?'
q1 = embed.encode(query)
len(q1)


384

In [3]:
# Answer 1
q1[0]

np.float64(-0.02058203437252893)

In [4]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [5]:
# Question 2
target = '02-vector-search/lessons/07-sqlitesearch-vector.md'
doc = next(d for d in documents if d['filename'] == target)
content = doc["content"]
content

'# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done so far is ex

In [6]:
# Embedding content
q2 = embed.encode(content)

# Calculating cosine similarity
q2.dot(q1)

np.float64(0.36107026789538205)

In [7]:
# Question 3
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [8]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
texts = [chunk.get("text", chunk.get("content", "")) if isinstance(chunk, dict)
         else getattr(chunk, "text", getattr(chunk, "content", "")) for chunk in chunks]
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    vectors.extend(batch_vectors)

len(vectors)

X = np.array(vectors)


  0%|          | 0/6 [00:00<?, ?it/s]

In [20]:
# Question 3
scores = X.dot(q1)
idx = np.argmax(scores)
doc = chunks[idx]
doc['filename']




'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [24]:
# Searching automatically using minsearch
from minsearch import VectorSearch
vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, chunks)

In [36]:
# Question 4

query = "What metric do we use to evaluate a search engine?"
query_vector = embed.encode(query)

results = vindex.search(query_vector, num_results=5)
print(results[0]['filename'])

04-evaluation/lessons/05-search-metrics.md


In [37]:
print([r['filename'] for r in results])

['04-evaluation/lessons/05-search-metrics.md', '04-evaluation/lessons/01-intro.md', '01-agentic-rag/lessons/05-search.md', '04-evaluation/lessons/01-intro.md', '04-evaluation/lessons/15-next-steps.md']


In [38]:
# Question 5
query = "How do I store vectors in PostgreSQL?"
query_vector = embed.encode(query)

results = vindex.search(query_vector, num_results=5)
print(results[0]['filename'])
print([r['filename'] for r in results])

02-vector-search/lessons/08-pgvector.md
['02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md']


In [35]:
from minsearch import Index
vindex2 = Index(text_fields=['content'])
vindex2.fit(chunks)

query = "How do I store vectors in PostgreSQL?"
results = vindex2.search(query, num_results=5)
print([r['filename'] for r in results])

['02-vector-search/lessons/02-embeddings.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/01-intro.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/01-intro.md']


In [ ]:
# Question 6

In [39]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [40]:
query = "How do I give the model access to tools?"
query_vector = embed.encode(query)

vector_results = vindex.search(query_vector, num_results=5)


In [41]:
from minsearch import Index
vindex2 = Index(text_fields=['content'])
vindex2.fit(chunks)

query = "How do I give the model access to tools?"
text_results = vindex2.search(query, num_results=5)

In [42]:
results = rrf([vector_results, text_results])
results

[{'start': 4000,
  'content': ' function. `parameters` is a JSON schema\nfor the arguments, and we mark `query` as required so the model always\nfills it in.\n\n## Sending the question with the tool\n\nNow we send the same question as before, but this time we include the\ntool in the request:\n\n```python\nresponse = openai_client.responses.create(\n    model="gpt-5.4-mini",\n    input=messages,\n    tools=[search_tool],\n)\n\nresponse.output\n```\n\nLook at the output. Instead of a message with the answer, the response\ncontains a `function_call` entry. The model decided it needs to search\nthe FAQ before answering. Rather than reply, it asked us to run the\nsearch function first.\n\nLook at the arguments too. The model didn\'t pass our question\nverbatim. It judged the raw question wasn\'t the best query to search\nwith. So it rewrote our enrollment question into search keywords like\n"enroll late join course".\n\n## Executing the function and sending the result back\n\nThe function 